In [1]:
import sys
print(sys.executable)


/home/aytu/apps/docsearch/.venv/bin/python


In [4]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [5]:
import os
from openai import OpenAI

client = OpenAI(api_key=api_key)
resp = client.responses.create(
    model="gpt-5.4-mini",
    input="Reply with: notebook setup works"
)
print(resp.output_text)

notebook setup works


In [9]:
import nest_asyncio
nest_asyncio.apply()

In [10]:
from pydantic_ai import Agent

agent = Agent(
    "openai:gpt-5.4-mini",
    system_prompt="You are a concise assistant."
)

In [12]:
result1 = agent.run_sync("My name is Ahmed.")
print(result1.output)

Nice to meet you, Ahmed.


In [16]:
import pandas as pd
pd.read_excel("/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx")

,company name,alternate name,link
0,mike,cold mike,C:\Users\maytu\OneDrive\Desktop\docs\1.docx
1,mike,hot mike,C:\Users\maytu\OneDrive\Desktop\docs\2.docx
2,ccc,osmanli,C:\Users\maytu\OneDrive\Desktop\docs\second_in...
3,ccc,selcuklu,C:\Users\maytu\OneDrive\Desktop\docs\second_in...


In [22]:
from pydantic import BaseModel
from typing import Optional, Literal

class WorkflowState(BaseModel):
    conversation_id: str
    original_user_query: str
    current_index_path: str
    current_level: int = 0

    awaiting_user_clarification: bool = False
    clarification_question: Optional[str] = None

    selected_company_name: Optional[str] = None
    selected_alternate_name: Optional[str] = None
    selected_link: Optional[str] = None

    status: Literal[
        "searching_index",
        "needs_clarification",
        "reading_sop",
        "resolved",
        "error"
    ] = "searching_index"

In [23]:
class IndexRow(BaseModel):
    row_id: int
    company_name: str
    alternate_name: str | None = None
    link: str

In [24]:
from typing import Literal
from pydantic import BaseModel

class RouteDecision(BaseModel):
    action: Literal["select_row", "ask_user", "no_match"]
    selected_row_id: int | None = None
    question: str | None = None
    reasoning_short: str | None = None

In [38]:
ROOT_INDEX_PATH = "/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx"

In [26]:
from pathlib import Path
import re

def normalize_path(raw_path: str) -> str:
    raw_path = raw_path.strip()

    if re.match(r"^[A-Za-z]:\\\\", raw_path):
        drive = raw_path[0].lower()
        rest = raw_path[2:].replace("\\\\", "/")
        return f"/mnt/{drive}{rest}"

    return str(Path(raw_path))

In [80]:
from pydantic import BaseModel, ConfigDict
from typing import Literal, Any

class ChatTurnResult(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    event_type: Literal["ask_user", "final_answer", "error"]
    assistant_text: str
    state: WorkflowState
    raw_result: Any = None

In [29]:
def handle_turn(user_input: str, state: WorkflowState | None) -> ChatTurnResult:
    if state is None:
        state = WorkflowState(
            conversation_id=new_id(),
            original_user_query=user_input,
            current_index_path=ROOT_INDEX_PATH,
            status="searching_index",
        )
    else:
        if state.awaiting_user_clarification:
            state.original_user_query += f"\nUser clarification: {user_input}"
            state.awaiting_user_clarification = False
            state.clarification_question = None

    while True:
        if state.status == "searching_index":
            rows = load_index_rows(state.current_index_path)

            decision = match_agent.run(
                user_query=state.original_user_query,
                rows=rows
            )

            if decision.action == "ask_user":
                state.awaiting_user_clarification = True
                state.clarification_question = decision.question
                state.status = "needs_clarification"
                return ask_user_result(decision.question, state)

            if decision.action == "no_match":
                state.status = "error"
                return error_result("Could not find a matching company.", state)

            selected_row = get_row_by_id(rows, decision.selected_row_id)
            selected_link = normalize_path(selected_row.link)

            state.selected_company_name = selected_row.company_name
            state.selected_alternate_name = selected_row.alternate_name
            state.selected_link = selected_link

            if selected_link.endswith(".docx"):
                state.status = "reading_sop"
                continue

            elif selected_link.endswith(".xlsx"):
                state.current_index_path = selected_link
                state.status = "searching_index"
                continue

            else:
                state.status = "error"
                return error_result("Unsupported linked file type.", state)

        elif state.status == "reading_sop":
            sop_text = parse_docx(state.selected_link)

            answer = qa_agent.run(
                user_query=state.original_user_query,
                sop_text=sop_text,
                company_name=state.selected_company_name,
            )

            state.status = "resolved"
            return final_answer_result(answer, state)

In [30]:
from pydantic import BaseModel
from pathlib import Path
import pandas as pd
import re

class IndexRow(BaseModel):
    row_id: int
    company_name: str
    alternate_name: str | None
    link: str

def normalize_windows_path(raw_path: str) -> str:
    raw_path = raw_path.strip()
    if re.match(r"^[A-Za-z]:\\", raw_path):
        drive = raw_path[0].lower()
        rest = raw_path[2:].replace("\\", "/")
        return f"/mnt/{drive}{rest}"
    return raw_path

def load_index_rows(index_path: str) -> list[IndexRow]:
    df = pd.read_excel(index_path) if index_path.endswith(".xlsx") else pd.read_csv(index_path)
    df = df.reset_index(drop=True)
    rows = []
    for i, row in df.iterrows():
        rows.append(
            IndexRow(
                row_id=i,
                company_name=str(row["company name"]).strip(),
                alternate_name=str(row["alternate name"]).strip() if pd.notna(row["alternate name"]) else None,
                link=str(row["link"]).strip(),
            )
        )
    return rows

In [31]:
from typing import Literal
from pydantic_ai import Agent

class RouteDecision(BaseModel):
    action: Literal["select_row", "ask_user", "no_match"]
    selected_row_id: int | None = None
    question: str | None = None

match_agent = Agent(
    "openai:gpt-4.1-mini",
    output_type=RouteDecision,
    system_prompt=(
        "You identify the best company row from the provided index rows. "
        "If multiple rows are plausible, ask exactly one short clarification question. "
        "If one row is clearly correct, return select_row."
    ),
)

/home/aytu/apps/docsearch/.venv/lib/python3.11/site-packages/pydantic_ai/agent/__init__.py:410: PydanticAIDeprecationWarning: In v2.0, 'openai:' will resolve to the OpenAI Responses API by default. Use 'openai-chat:' to keep current Chat Completions behavior, or 'openai-responses:' to opt in early.
  self._model = models.infer_model(model)


In [ ]:
def build_match_prompt(user_input: str, rows: list[IndexRow]) -> str:
    row_text = "\n".join(
        f"row_id={r.row_id} | company={r.company_name} | alternate={r.alternate_name} | link={r.link}"
        for r in rows
    )
    return f"""
User query: {user_input}

Index rows:
{row_text}

Choose one:
- select_row
- ask_user
- no_match
"""

# async def handle_turn(user_input: str, state: WorkflowState | None) -> ChatTurnResult:
    if state is None:
        state = WorkflowState(
            conversation_id="demo-1",
            original_user_query=user_input,
            current_index_path=ROOT_INDEX_PATH,
            status="searching_index",
        )
    elif state.awaiting_user_clarification:
        state.original_user_query += f"\nUser clarification: {user_input}"
        state.awaiting_user_clarification = False
        state.clarification_question = None

    while True:
        rows = load_index_rows(state.current_index_path)
        table_text = dataframe_to_row_text(df)

        prompt = f"""
        User query:
        {user_input}

        Current table:
        {table_text}

        Your task:
        - Choose the best matching row if clear.
        - If ambiguous, ask one short clarification question.
        - If no row matches, return no_match.
        """
        decision = (await match_agent.run(build_match_prompt(state.original_user_query, rows))).output

        if decision.action == "ask_user":
            state.awaiting_user_clarification = True
            state.clarification_question = decision.question
            state.status = "needs_clarification"
            return ChatTurnResult(
                event_type="ask_user",
                assistant_text=decision.question,
                state=state,
            )

        if decision.action == "no_match":
            state.status = "error"
            return ChatTurnResult(
                event_type="error",
                assistant_text="I could not identify the company from the index.",
                state=state,
            )

        selected = next(r for r in rows if r.row_id == decision.selected_row_id)
        resolved_link = normalize_windows_path(selected.link)

        if resolved_link.lower().endswith(".docx"):
            state.selected_company_name = selected.company_name
            state.selected_link = resolved_link
            state.status = "reading_sop"

            sop_result = await answer_from_sop(state.original_user_query, state.selected_link)
            state.status = "resolved"

            return ChatTurnResult(
                event_type="final_answer",
                assistant_text=sop_result.answer,
                state=state,
            )

        if resolved_link.lower().endswith(".xlsx"):
            state.current_index_path = resolved_link
            state.status = "searching_index"
            continue

        state.status = "error"
        return ChatTurnResult(
            event_type="error",
            assistant_text="The selected link is neither a DOCX SOP nor an XLSX index.",
            state=state,
        )

    return ChatTurnResult(
        event_type="final_answer",
        assistant_text=f"Resolved SOP path: {state.selected_link}",
        state=state,
    )

In [ ]:
# async def handle_turn(user_input: str, state: WorkflowState | None) -> ChatTurnResult:
#     if state is None:
#         state = WorkflowState(
#             conversation_id="demo-1",
#             original_user_query=user_input,
#             current_index_path=ROOT_INDEX_PATH,
#             status="searching_index",
#         )
#     elif state.awaiting_user_clarification:
#         state.original_user_query += f"\nUser clarification: {user_input}"
#         state.awaiting_user_clarification = False
#         state.clarification_question = None

#     while True:
#         df = load_small_table(state.current_index_path)
#         table_text = dataframe_to_row_text(df)

#         prompt = f"""
# User query:
# {state.original_user_query}

# Current table:
# {table_text}

# Your task:
# - Choose the best matching row if clear.
# - If ambiguous, ask one short clarification question.
# - If no row matches, return no_match.
# """

#         decision = (await match_agent.run(prompt)).output

#         if decision.action == "ask_user":
#             state.awaiting_user_clarification = True
#             state.clarification_question = decision.question
#             state.status = "needs_clarification"
#             return ChatTurnResult(
#                 event_type="ask_user",
#                 assistant_text=decision.question,
#                 state=state,
#                 raw_result=agent_result,
#             )

#         if decision.action == "no_match":
#             state.status = "error"
#             return ChatTurnResult(
#                 event_type="error",
#                 assistant_text="I could not identify the right row in the table.",
#                 state=state,
#             )

#         selected_row = df.iloc[decision.selected_row_id]
#         link = extract_link_from_row(selected_row)
#         resolved_link = normalize_windows_path(link)

#         if resolved_link.lower().endswith(".xlsx"):
#             state.current_index_path = resolved_link
#             state.status = "searching_index"
#             continue

#         if resolved_link.lower().endswith(".docx"):
#             state.selected_link = resolved_link
#             state.status = "reading_sop"

#             sop_result = await answer_from_sop(state.original_user_query, state.selected_link)
#             state.status = "resolved"

#             return ChatTurnResult(
#                 event_type="final_answer",
#                 assistant_text=sop_result.answer,
#                 state=state,
#             )

#         state.status = "error"
#         return ChatTurnResult(
#             event_type="error",
#             assistant_text="Selected row did not contain a usable DOCX or XLSX link.",
#             state=state,
#         )

In [83]:
async def handle_turn(
    user_input: str,
    state: WorkflowState | None,
    message_history=None,
) -> ChatTurnResult:
    if message_history is None:
        message_history = []

    if state is None:
        state = WorkflowState(
            conversation_id="demo-1",
            original_user_query=user_input,
            current_index_path=ROOT_INDEX_PATH,
            status="searching_index",
        )
    elif state.awaiting_user_clarification:
        state.original_user_query += f"\nUser clarification: {user_input}"
        state.awaiting_user_clarification = False
        state.clarification_question = None
    else:
        state.original_user_query = user_input

    while True:
        df = load_small_table(state.current_index_path)
        table_text = dataframe_to_row_text(df)

        prompt = f"""
User query:
{state.original_user_query}

Current table:
{table_text}

Your task:
- Choose the best matching row if clear.
- If ambiguous, ask one short clarification question.
- If no row matches, return no_match.
"""

        agent_result = await match_agent.run(
            prompt,
            message_history=message_history,
        )
        decision = agent_result.output

        if decision.action == "ask_user":
            state.awaiting_user_clarification = True
            state.clarification_question = decision.question
            state.status = "needs_clarification"

            return ChatTurnResult(
                event_type="ask_user",
                assistant_text=decision.question,
                state=state,
                raw_result=agent_result,
            )

        if decision.action == "no_match":
            state.status = "error"

            return ChatTurnResult(
                event_type="error",
                assistant_text="I could not identify the right row in the table.",
                state=state,
                raw_result=agent_result,
            )

        selected_row = df.iloc[decision.selected_row_id]
        link = extract_link_from_row(selected_row)

        if not link:
            state.status = "error"
            return ChatTurnResult(
                event_type="error",
                assistant_text="The selected row did not contain a usable file link.",
                state=state,
                raw_result=agent_result,
            )

        resolved_link = normalize_windows_path(link)

        if resolved_link.lower().endswith(".xlsx"):
            state.current_index_path = resolved_link
            state.status = "searching_index"
            message_history = agent_result.all_messages()
            continue

        if resolved_link.lower().endswith(".docx"):
            state.selected_link = resolved_link
            state.status = "reading_sop"

            sop_result = await answer_from_sop(
                state.original_user_query,
                state.selected_link,
            )
            state.status = "resolved"

            return ChatTurnResult(
                event_type="final_answer",
                assistant_text=sop_result.answer,
                state=state,
                raw_result=agent_result,
            )

        state.status = "error"
        return ChatTurnResult(
            event_type="error",
            assistant_text="Selected row did not contain a usable DOCX or XLSX link.",
            state=state,
            raw_result=agent_result,
        )

In [33]:
class SopAnswer(BaseModel):
    answer: str

qa_agent = Agent(
    "openai:gpt-4.1-mini",
    output_type=SopAnswer,
    system_prompt=(
        "Answer the user's question only from the SOP text provided. "
        "If the SOP does not contain the answer, say that clearly."
    ),
)

In [62]:
def load_small_table(index_path: str):
    path = Path(index_path)

    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path, engine="openpyxl")
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    df.columns = [str(c).strip() for c in df.columns]
    return df

def dataframe_to_row_text(df: pd.DataFrame) -> str:
    lines = []
    for i, row in df.reset_index(drop=True).iterrows():
        cells = [f"{col}={row[col]}" for col in df.columns]
        lines.append(f"row_id={i} | " + " | ".join(cells))
    return "\n".join(lines)

In [63]:
class TableDecision(BaseModel):
    action: Literal["select_row", "ask_user", "no_match"]
    selected_row_id: int | None = None
    question: str | None = None

In [64]:
def extract_link_from_row(row: pd.Series) -> str | None:
    for value in row.values:
        if isinstance(value, str):
            v = value.strip()
            if v.lower().endswith(".docx") or v.lower().endswith(".xlsx"):
                return v
    return None

In [35]:
from docx import Document

def read_docx_text(path: str) -> str:
    doc = Document(path)
    return "\n".join(p.text for p in doc.paragraphs if p.text.strip())

In [55]:
from docx import Document
from pydantic import BaseModel
from pydantic_ai import Agent

def read_docx_text(path: str) -> str:
    doc = Document(path)
    return "\n".join(
        p.text.strip()
        for p in doc.paragraphs
        if p.text and p.text.strip()
    )

class SopAnswer(BaseModel):
    answer: str

qa_agent = Agent(
    "openai:gpt-4.1-mini",
    output_type=SopAnswer,
    system_prompt=(
        "Answer the user's question only from the SOP text provided. "
        "If the answer is not in the SOP, say that clearly."
    ),
)

async def answer_from_sop(user_question: str, sop_path: str):
    sop_text = read_docx_text(sop_path)

    prompt = f"""
User question:
{user_question}

SOP text:
{sop_text}
"""
    result = await qa_agent.run(prompt)
    return result.output

/home/aytu/apps/docsearch/.venv/lib/python3.11/site-packages/pydantic_ai/agent/__init__.py:410: PydanticAIDeprecationWarning: In v2.0, 'openai:' will resolve to the OpenAI Responses API by default. Use 'openai-chat:' to keep current Chat Completions behavior, or 'openai-responses:' to opt in early.
  self._model = models.infer_model(model)


In [67]:
# --- test conversation cell ---

state = None

async def chat_once(user_text: str):
    global state
    result = await handle_turn(user_text, state)
    state = result.state

    print(f"\nUSER: {user_text}")
    print(f"BOT ({result.event_type}): {result.assistant_text}")
    print(f"STATUS: {state.status}")

    if getattr(state, "current_index_path", None):
        print(f"CURRENT INDEX: {state.current_index_path}")

    if getattr(state, "selected_link", None):
        print(f"SELECTED LINK: {state.selected_link}")

    return result

In [51]:
await chat_once("I need the SOP for mike")


USER: I need the SOP for mike
BOT (ask_user): Do you need the SOP for 'cold mike' or 'hot mike'?
STATUS: needs_clarification
CURRENT INDEX: /mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx


ChatTurnResult(event_type='ask_user', assistant_text="Do you need the SOP for 'cold mike' or 'hot mike'?", state=WorkflowState(conversation_id='demo-1', original_user_query='I need the SOP for mike', current_index_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx', current_level=0, awaiting_user_clarification=True, clarification_question="Do you need the SOP for 'cold mike' or 'hot mike'?", selected_company_name=None, selected_alternate_name=None, selected_link=None, status='needs_clarification'))

In [68]:
await chat_once("I need the SOP for ccc")


USER: I need the SOP for ccc
BOT (ask_user): Do you want the SOP for 'osmanli' or 'selcuklu' under ccc?
STATUS: needs_clarification
CURRENT INDEX: /mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx


ChatTurnResult(event_type='ask_user', assistant_text="Do you want the SOP for 'osmanli' or 'selcuklu' under ccc?", state=WorkflowState(conversation_id='demo-1', original_user_query='I need the SOP for ccc', current_index_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx', current_level=0, awaiting_user_clarification=True, clarification_question="Do you want the SOP for 'osmanli' or 'selcuklu' under ccc?", selected_company_name=None, selected_alternate_name=None, selected_link=None, status='needs_clarification'))

In [70]:
result = await chat_once("i know the address. would that help?")


USER: i know the address. would that help?
BOT (ask_user): Can you please specify which alternate name for CCC you are referring to: Osmanli or Selcuklu?
STATUS: needs_clarification
CURRENT INDEX: /mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx


In [76]:
result = await agent.run("Si know the address. would that help?", message_history=message_history)

# Python-native message objects
history = result.all_messages()
for msg in history:
    print(msg.role, ":", msg.content)

# JSON-serializable version (e.g., to store or inspect)
history_json = result.all_messages_json()
print(history_json)

NameError: name 'message_history' is not defined

In [77]:
message_history = []

async def ask(text: str):
    global message_history
    result = await agent.run(text, message_history=message_history)
    message_history = result.all_messages()
    print(result.output)
    return result

In [78]:
await ask("I need the SOP for ccc")

Could you clarify what “ccc” stands for?

For example, it could mean:
- a company, team, or department name
- a specific process/system
- “cost center code”
- “customer care center”
- something else

If you want, I can draft a standard SOP template for CCC right away if you tell me:
1. what CCC stands for  
2. the purpose of the SOP  
3. the process/workflow it should cover


AgentRunResult(output='Could you clarify what “ccc” stands for?\n\nFor example, it could mean:\n- a company, team, or department name\n- a specific process/system\n- “cost center code”\n- “customer care center”\n- something else\n\nIf you want, I can draft a standard SOP template for CCC right away if you tell me:\n1. what CCC stands for  \n2. the purpose of the SOP  \n3. the process/workflow it should cover')

In [75]:
print(result.state.model_dump_json())

{"conversation_id":"demo-1","original_user_query":"I need the SOP for ccc\nUser clarification: i know the address. would that help?\nUser clarification: i know the address. would that help?","current_index_path":"/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx","current_level":0,"awaiting_user_clarification":true,"clarification_question":"Can you please specify which alternate name for CCC you are referring to: Osmanli or Selcuklu?","selected_company_name":null,"selected_alternate_name":null,"selected_link":null,"status":"needs_clarification"}


In [58]:
await chat_once("what does sop tell me")


USER: what does sop tell me
BOT (final_answer): The SOP for hot mike is to use the red cable.
STATUS: resolved
CURRENT INDEX: /mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx
SELECTED LINK: /mnt/c/Users/maytu/OneDrive/Desktop/docs/2.docx


ChatTurnResult(event_type='final_answer', assistant_text='The SOP for hot mike is to use the red cable.', state=WorkflowState(conversation_id='demo-1', original_user_query='I need the SOP for hot mike', current_index_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx', current_level=0, awaiting_user_clarification=False, clarification_question=None, selected_company_name='mike', selected_alternate_name=None, selected_link='/mnt/c/Users/maytu/OneDrive/Desktop/docs/2.docx', status='resolved'))

In [84]:
state = None
message_history = []
last_result = None

async def chat_once(user_text: str):
    global state, message_history, last_result

    result = await handle_turn(
        user_text,
        state,
        message_history=message_history,
    )

    state = result.state
    last_result = result.raw_result

    if last_result is not None:
        message_history = last_result.all_messages()

    print(f"\nUSER: {user_text}")
    print(f"BOT ({result.event_type}): {result.assistant_text}")
    print(f"STATUS: {state.status}")
    print(f"HISTORY LEN: {len(message_history)}")

    return result



In [85]:
prompt = "I need the SOP for ccc"

agent_result = await match_agent.run(prompt, message_history=message_history)
decision = agent_result.output

In [87]:
await chat_once("I need the SOP for ccc")
await chat_once("selcuklu")
print(last_result.all_messages_json())



USER: I need the SOP for ccc
BOT (ask_user): Do you need the SOP for the 'osmanli' or 'selcuklu' alternate name of ccc?
STATUS: needs_clarification
HISTORY LEN: 3

USER: selcuklu
BOT (final_answer): The SOP for Selcuklu company is to use key 1071.
STATUS: resolved
HISTORY LEN: 8
b'[{"parts":[{"content":"You identify the best company row from the provided index rows. If multiple rows are plausible, ask exactly one short clarification question. If one row is clearly correct, return select_row.","timestamp":"2026-05-26T20:16:02.056231Z","dynamic_ref":null,"part_kind":"system-prompt"},{"content":"\\nUser query:\\nI need the SOP for ccc\\n\\nCurrent table:\\nrow_id=0 | company name=mike | alternate name=cold mike | link=C:\\\\Users\\\\maytu\\\\OneDrive\\\\Desktop\\\\docs\\\\1.docx\\nrow_id=1 | company name=mike | alternate name=hot mike | link=C:\\\\Users\\\\maytu\\\\OneDrive\\\\Desktop\\\\docs\\\\2.docx\\nrow_id=2 | company name=ccc | alternate name=osmanli | link=C:\\\\Users\\\\maytu\\\\

In [94]:
import gradio as gr

def init_session():
    return {
        "workflow_state": None,
        "message_history": [],
    }

async def respond(message, chat_history, session):
    if not message or not message.strip():
        return chat_history, session, {}, ""

    if session is None:
        session = init_session()

    result = await handle_turn(
        user_input=message,
        state=session["workflow_state"],
        message_history=session["message_history"],
    )

    session["workflow_state"] = result.state

    if result.raw_result is not None:
        session["message_history"] = result.raw_result.all_messages()

    chat_history = chat_history or []
    chat_history = chat_history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": result.assistant_text},
    ]

    debug_payload = {
        "event_type": result.event_type,
        "assistant_text": result.assistant_text,
        "workflow_state": result.state.model_dump(),
        "message_count": len(session["message_history"]),
    }

    return chat_history, session, debug_payload, ""

def reset_session():
    return [], init_session(), {}

with gr.Blocks() as demo:
    gr.Markdown("## SOP Bot")

    chatbot = gr.Chatbot(height=500)
    msg = gr.Textbox(label="Ask about a company or SOP", placeholder="Ask a question...")
    send = gr.Button("Send")
    clear = gr.Button("Reset")
    session = gr.State(init_session())
    debug = gr.JSON(label="Debug State")

    send.click(
        respond,
        inputs=[msg, chatbot, session],
        outputs=[chatbot, session, debug, msg],
    )

    msg.submit(
        respond,
        inputs=[msg, chatbot, session],
        outputs=[chatbot, session, debug, msg],
    )

    clear.click(
        reset_session,
        inputs=[],
        outputs=[chatbot, session, debug],
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [90]:
import gradio as gr
print(gr.__version__)

6.14.0
